# 第38章 统计柱状图（barplot）

用barplot比较分类组的均值或其他估计量，并理解误差线。

## 学习目标

本章围绕一种明确的图表结构展开，先看最小可用示例，再加入分组、注释或交互细节。


## 适用场景

比较各类别的平均值、中位数或自定义统计量。

## 数据结构

一列类别和一列数值；每组需要多个观察才能估计误差。

## 本章练习任务

运行基础图表后，完成以下任务：

1. 将 estimator="mean" 改为 estimator="median"，对比均值与中位数的柱高差异
2. 修改 errorbar=("ci", 90) 为 errorbar="sd"，观察置信区间与标准差的误差线长度
3. 将 errorbar=None 改为 errorbar=("ci", 95)，说明误差线对估计不确定性的表达作用


## 0. 准备可复现数据

先完成导入和数据准备，后续单元格只负责一种图表或一种分析动作。


In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

rng = np.random.default_rng(36)
n = 240
orders = pd.DataFrame({
    "category": rng.choice(["办公", "数码", "家居"], n, p=[0.34, 0.38, 0.28]),
    "channel": rng.choice(["自然流量", "广告", "会员"], n, p=[0.42, 0.36, 0.22]),
    "region": rng.choice(["华东", "华南", "华北"], n),
    "order_value": np.clip(rng.normal(260, 72, n), 45, None),
    "items": rng.integers(1, 7, n),
})
orders.loc[orders["category"] == "数码", "order_value"] *= 1.35
orders["satisfied"] = rng.choice(["满意", "一般"], n, p=[0.78, 0.22])

marketing = pd.DataFrame({
    "channel": rng.choice(["搜索", "社交", "会员"], n),
    "visits": rng.integers(80, 850, n),
    "ad_spend": rng.uniform(2, 38, n),
})
marketing["sales"] = (
    45 + marketing["visits"] * 0.16 + marketing["ad_spend"] * 2.4
    + marketing["channel"].map({"搜索": 18, "社交": 8, "会员": 32})
    + rng.normal(0, 28, n)
).clip(10)
marketing["conversion"] = (marketing["sales"] / marketing["visits"]).clip(0.02, 0.5)

daily = pd.DataFrame({
    "date": np.tile(pd.date_range("2026-01-01", periods=12, freq="D"), 3),
    "region": np.repeat(["华东", "华南", "华北"], 12),
})
daily["sales"] = (
    np.tile(np.linspace(110, 190, 12), 3)
    + np.repeat([28, 8, 18], 12)
    + rng.normal(0, 9, 36)
)

sns.set_theme(style="whitegrid", context="notebook")
print("订单样本:", orders.shape, "营销样本:", marketing.shape)


## 1. 基础图表

先保留必要的编码：位置、颜色或大小。图表标题、坐标轴和单位应能让读者脱离代码理解结果。


In [ ]:
summary = orders.groupby("category")["order_value"].agg(["mean", "median", "count"]).round(1)
display(summary)
fig, ax = plt.subplots(figsize=(8, 4.2))
sns.barplot(data=orders, x="category", y="order_value", errorbar=None, color="#1a73e8", ax=ax)
ax.set(title="品类平均客单价", xlabel="品类", ylabel="平均客单价（元）")
fig.tight_layout()
plt.show()


## 2. 进阶变体

在基础图表可读的前提下增加分组、布局、注释或交互。新增编码必须服务于一个明确问题。


In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.5))
sns.barplot(data=orders, x="category", y="order_value", hue="channel", estimator="mean", errorbar=("ci", 90), palette="colorblind", ax=ax)
ax.set(title="分渠道比较品类客单价", xlabel="品类", ylabel="平均客单价（元）")
ax.legend(title="渠道", frameon=False)
fig.tight_layout()
plt.show()


## 3. 参数说明

- estimator：估计量
- errorbar：误差表示
- hue：分组
- order：顺序


## 4. 结果解读

柱高是估计值，误差线含义由errorbar参数决定；同时报告样本量。


## 常见误区

- 把均值柱高解释为总量
- 隐藏分布和样本量
- 误差线含义不明确


## 综合练习

请使用同一份数据完成下面任务，并说明你选择该图表的原因。


In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.2))
sns.barplot(data=orders, x="region", y="items", estimator="mean", errorbar="sd", color="#188038", ax=ax)
ax.set(title="区域平均购买件数及标准差", xlabel="区域", ylabel="件数")
fig.tight_layout()
plt.show()


## 本章小结

用barplot比较分类组的均值或其他估计量，并理解误差线。


### 你已经掌握

- 判断统计柱状图（barplot）的适用场景
- 准备与图表匹配的数据结构
- 从基础图表扩展到分组、注释或交互变体
- 按照业务问题解读图表并说明结论边界


### 图表选择速查

| 选择要点 | 本章说明 |
| --- | --- |
| 适用场景 | 比较各类别的平均值、中位数或自定义统计量。 |
| 数据结构 | 一列类别和一列数值；每组需要多个观察才能估计误差。 |
| 结果解读 | 柱高是估计值，误差线含义由errorbar参数决定；同时报告样本量。 |


### 关键参数

| 参数 | 作用 |
| --- | --- |
| `estimator` | 估计量 |
| `errorbar` | 误差表示 |
| `hue` | 分组 |
| `order` | 顺序 |


### 需要注意

- 把均值柱高解释为总量
- 隐藏分布和样本量
- 误差线含义不明确


### 完成检查

- [ ] 能判断什么问题适合使用统计柱状图（barplot）
- [ ] 能准备符合要求的数据结构
- [ ] 能独立完成基础图表和一个进阶变体
- [ ] 能调整关键参数并解释视觉变化
- [ ] 能根据图表写出有边界的数据结论
